# 03 — Single-Market Resolution Analysis

**Question:** Of all single-outcome markets Polymarket launches, what fraction resolve YES vs NO?

**Date:** 2026-03-25
**Data:** Full population of resolved single-market events from Polymarket Gamma API (no volume filter)

---

## Why this matters

Polymarket frames its single-outcome markets as binary YES/NO questions. If these markets were symmetric ("Will Team A or Team B win?"), we'd expect roughly 50/50 resolution. But most markets are directional ("Will X happen by date Y?"), which could introduce a structural skew in how often YES vs NO wins.

Understanding the base rate of YES vs NO resolution is important for anyone interpreting prediction market prices, building trading strategies, or studying market calibration.

**Data source:** `fetch_resolution_population()` in `src/fetch.py` — paginates through all resolved events on the Gamma API, no trade data needed. Cached at `data/resolution_population.parquet`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

%matplotlib inline

sys.path.insert(0, str(Path("..").resolve()))
from src.fetch import fetch_resolution_population
from src.config import OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
})

In [ ]:
df = fetch_resolution_population(force_refresh=False)

n = len(df)
yes = int(df["resolved_yes"].sum())
no = n - yes
p = yes / n

print(f"Population: {n:,} resolved single-market events")
print(f"Resolved YES: {yes:,} ({p*100:.1f}%)")
print(f"Resolved NO:  {no:,} ({(1-p)*100:.1f}%)")

---
## Population-level resolution split

Is the YES/NO resolution split 50/50 across all single-market events?

In [ ]:
# Binomial test: H0 = 50/50 resolution
bt = stats.binomtest(yes, n, p=0.5, alternative="two-sided")

# Wilson score 95% CI
z = 1.96
denom = 1 + z**2 / n
center = (p + z**2 / (2 * n)) / denom
spread = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
ci_low = max(0, center - spread)
ci_high = min(1, center + spread)

print(f"YES resolution rate: {p*100:.1f}%")
print(f"95% CI: [{ci_low*100:.1f}%, {ci_high*100:.1f}%]")
print(f"Binomial test vs 50%: p = {bt.pvalue:.2e}")
print(f"")
print(f"Result: Single-market events resolve NO {(1-p)*100:.1f}% of the time.")
print(f"A naive NO bet wins ~60% of the time.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(["YES", "NO"], [yes, no], color=["#4ECDC4", "#C44D58"], width=0.5)
ax.set_ylabel("Events")
ax.set_title(f"Single-market event resolution (n={n:,})")

for bar, count, pct in zip(bars, [yes, no], [p, 1-p]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{count:,}\n({pct*100:.1f}%)", ha="center", fontweight="bold")

ax.axhline(n/2, color="gray", ls="--", alpha=0.4, label="50/50 line")
ax.legend()
ax.set_ylim(0, max(yes, no) * 1.15)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "16-resolution-population.png"), dpi=150, bbox_inches="tight")

### Interpretation

Across the full population of resolved single-market events, **NO wins ~59% of the time.** This is not close to 50/50 — the binomial test rejects the null with extreme confidence.

This is a question design effect: Polymarket predominantly creates markets of the form "Will [specific thing] happen?" Specific things usually don't happen by specific dates. The asymmetry is baked into how questions are framed, not into how markets are priced.

---
## Resolution split by category

Is the NO skew uniform, or do some categories drive it?

In [ ]:
cat_stats = df.groupby("category").agg(
    total=("resolved_yes", "count"),
    yes=("resolved_yes", "sum"),
).assign(
    no=lambda x: x["total"] - x["yes"],
    yes_pct=lambda x: x["yes"] / x["total"] * 100,
    no_pct=lambda x: (x["total"] - x["yes"]) / x["total"] * 100,
).sort_values("total", ascending=False)

# Add 95% Wilson CI for each category
z = 1.96
cat_stats["ci_low"] = cat_stats.apply(
    lambda r: max(0, ((r["yes"]/r["total"]) + z**2/(2*r["total"])) / (1 + z**2/r["total"])
              - z * np.sqrt(((r["yes"]/r["total"])*(1-r["yes"]/r["total"]) + z**2/(4*r["total"])) / r["total"]) / (1 + z**2/r["total"])) * 100,
    axis=1
)
cat_stats["ci_high"] = cat_stats.apply(
    lambda r: min(100, ((r["yes"]/r["total"]) + z**2/(2*r["total"])) / (1 + z**2/r["total"])
              + z * np.sqrt(((r["yes"]/r["total"])*(1-r["yes"]/r["total"]) + z**2/(4*r["total"])) / r["total"]) / (1 + z**2/r["total"])) * 100,
    axis=1
)

print(f"{'Category':<12} {'Total':>6} {'YES':>6} {'NO':>6} {'YES%':>7} {'NO%':>7}  {'95% CI (YES)'}")
print("-" * 75)
for _, r in cat_stats.iterrows():
    print(f"{r.name:<12} {int(r['total']):>6} {int(r['yes']):>6} {int(r['no']):>6} {r['yes_pct']:>6.1f}% {r['no_pct']:>6.1f}%  [{r['ci_low']:.1f}%, {r['ci_high']:.1f}%]")
print("-" * 75)
print(f"{'TOTAL':<12} {int(cat_stats['total'].sum()):>6} {int(cat_stats['yes'].sum()):>6} {int(cat_stats['no'].sum()):>6} {p*100:>6.1f}% {(1-p)*100:>6.1f}%")

In [ ]:
# Only plot categories with 30+ events for visual clarity
plot_cats = cat_stats[cat_stats["total"] >= 30].copy()
plot_cats = plot_cats.sort_values("yes_pct", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))

y_pos = range(len(plot_cats))
bars = ax.barh(y_pos, plot_cats["yes_pct"], color="#4ECDC4", height=0.6)

# Error bars from CI
xerr_low = plot_cats["yes_pct"].values - plot_cats["ci_low"].values
xerr_high = plot_cats["ci_high"].values - plot_cats["yes_pct"].values
ax.errorbar(plot_cats["yes_pct"], y_pos, xerr=[xerr_low, xerr_high],
            fmt="none", color="black", capsize=3, linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels([f"{cat} (n={int(row['total'])})"
                     for cat, row in plot_cats.iterrows()])
ax.set_xlabel("YES resolution rate (%)")
ax.set_title("YES resolution rate by category — single-market events")
ax.axvline(50, color="gray", ls="--", alpha=0.5, label="50%")
ax.axvline(p * 100, color="#C44D58", ls="-", alpha=0.7, label=f"Population avg ({p*100:.0f}%)")
ax.legend(loc="lower right")
ax.set_xlim(0, 60)

# Add percentage labels
for i, (_, row) in enumerate(plot_cats.iterrows()):
    ax.text(row["yes_pct"] + 1.5, i, f"{row['yes_pct']:.0f}%", va="center", fontweight="bold")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "17-resolution-by-category.png"), dpi=150, bbox_inches="tight")

### Category breakdown — interpretation

The NO skew is not uniform. There's a clear spectrum tied to question framing:

**Near 50/50 — Sports (~47% YES).** Sports markets are often symmetric: "Will Team A beat Team B?" is structurally close to a coin flip. This is the only category near parity.

**Moderate NO skew — Politics (~31%), Crypto (~35%), Finance (~38%).** These tend to be directional: "Will X happen by date Y?" Things often don't happen by specific dates.

**Heavy NO skew — Culture (~21%), Science (~23%), Tech (~19%), World (~7%).** Heavily directional/aspirational questions where the default outcome is "no, that didn't happen." World is the most extreme: 28 of 30 resolved NO (e.g., "Russia x Ukraine ceasefire before July?", "Trump takes Panama Canal in 2025?"). These are low-base-rate events by design — Polymarket creates them because they attract speculative volume, not because they're 50/50 propositions. World is too small (n=30) to draw robust conclusions from individually, but it illustrates the pattern.

**Unknown category (n≈2,900, 45% YES)** is older markets from before Polymarket had consistent tagging — a mix of 2021-era NBA spreads, early COVID markets, 2020 election markets, and early crypto price markets. The near-parity YES rate is consistent with a grab bag heavy on sports spreads.

**The pattern:** The more directional the question framing ("Will [specific thing] happen?"), the more it resolves NO. Categories with symmetric framing (Sports) sit near 50/50.

---
## Does volume correlate with resolution?

Do higher-volume markets resolve differently than lower-volume ones?

In [ ]:
# Split into volume quartiles
df["volume_quartile"] = pd.qcut(df["volume"], q=4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])

vol_res = df.groupby("volume_quartile", observed=True).agg(
    total=("resolved_yes", "count"),
    yes=("resolved_yes", "sum"),
    median_volume=("volume", "median"),
).assign(
    yes_pct=lambda x: x["yes"] / x["total"] * 100,
)

print(f"{'Quartile':<15} {'Events':>7} {'YES%':>7} {'Median Vol':>12}")
print("-" * 45)
for q, row in vol_res.iterrows():
    print(f"{q:<15} {int(row['total']):>7} {row['yes_pct']:>6.1f}% ${row['median_volume']:>10,.0f}")

# Correlation test
corr, corr_p = stats.pointbiserialr(df["resolved_yes"].astype(int), np.log1p(df["volume"]))
print(f"\nPoint-biserial correlation (log volume vs resolution): r={corr:.3f}, p={corr_p:.2e}")

### Volume and resolution

Higher-volume markets tend to resolve YES more often. This likely reflects a selection effect: events that *do* happen generate more trading activity and attention, while speculative "will X happen?" markets that quietly resolve NO attract less volume.

This means any analysis that samples by volume (e.g., top-N markets) will overweight YES-resolving events relative to the population.

---
## Headline finding

**On Polymarket, betting NO on single-outcome markets wins ~60% of the time.**

This is a question design effect, not a market inefficiency. Polymarket predominantly frames markets as "Will [specific thing] happen?" Specific things usually don't happen. The NO skew ranges from modest (Sports: 53%) to extreme (World/Tech: 80-93%).

**Key caveats:**
- This says nothing about risk-adjusted returns — NO tokens at $0.90 pay much less than YES tokens at $0.10
- The profitable question is not "does NO win more often?" but "are NO tokens priced correctly given that NO wins more often?"
- Sports markets are near 50/50 — the skew is concentrated in directional/aspirational categories
- This is resolution *frequency*, not P&L. Frequency alone does not imply a profitable strategy
- ~2,900 events lack category tags (pre-tagging era) — these sit at ~45% YES and are likely a sports-heavy mix